# Retrieval Augmented Generation (RAG) and Vector Databases with OpenAI

In [2]:
import os 
import pandas as pd  
import numpy as np 
from openai import OpenAI

## Initializing the OpenAI Client

In [3]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
Embedding_model="text-embedding-3-small"

chat_model="gpt-4o-mini"

In [4]:
data_paths=["data/frameworks.md","data/own_framework.md"]

rows =[]

for path in data_paths:
    with open(path,'r' ,encoding='utf-8') as file:
        file_content=file.read()

    
    rows.append({'path': path, 'text': file_content})

df = pd.DataFrame(rows, columns=['path','text'])

df.head

<bound method NDFrame.head of                     path                                               text
0     data/frameworks.md  # Neural Network Frameworks\n\nAs we have lear...
1  data/own_framework.md  # Introduction to Neural Networks. Multi-Layer...>

In [5]:
def split_text(text,max_length=400, min_length=300):
    words=text.split()
    chunks=[]
    current_chunk=[]

    for word in words :
        current_chunk.append(word)
        if len(''.join(current_chunk))>max_length and len(''.join(current_chunk))>min_length:
            chunks.append(''.join(current_chunk))
            current_chunk=[]

    if current_chunk:
        chunks.append(''.join(current_chunk))

    return chunks 

splitted_df=df.copy()
splitted_df['chunks']=splitted_df['text'].apply(lambda x : split_text(x,400,300))

splitted_df.head()

,path,text,chunks
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,[#NeuralNetworkFrameworksAswehavelearnedalread...
1,data/own_framework.md,# Introduction to Neural Networks. Multi-Layer...,[#IntroductiontoNeuralNetworks.Multi-LayeredPe...


In [6]:
flattened_df=splitted_df.explode('chunks').reset_index(drop=True)
flattened_df.head()

,path,text,chunks
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,#NeuralNetworkFrameworksAswehavelearnedalready...
1,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,hadtomanuallyprogramallderivativefunctionsinsi...
2,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,>✅Theterm'parallelize'meanstodistributethecomp...
3,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,High-levelAPI|Keras|Pytorch**Low-levelAPIs**in...
4,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,"neuralnetworksasa**sequenceoflayers**,andmakec..."


## Converting Text to Embeddings

In [7]:
def create_embeddings(text,model=Embedding_model):

    response = client.embeddings.create(input=[text],model=model)

    return response.data[0].embedding

sample_embedding = create_embeddings("Hello world")

print(f"Embedding dimension:{len(sample_embedding)}")

Embedding dimension:1536


In [8]:
embeddings_list=[]
for chunk in flattened_df['chunks']:
    embeddings_list.append(create_embeddings(chunk))


flattened_df['embeddings']=embeddings_list

flattened_df.head()

,path,text,chunks,embeddings
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,#NeuralNetworkFrameworksAswehavelearnedalready...,"[0.01087188720703125, -0.0132904052734375, 0.0..."
1,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,hadtomanuallyprogramallderivativefunctionsinsi...,"[0.0180816650390625, -0.025421142578125, 0.057..."
2,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,>✅Theterm'parallelize'meanstodistributethecomp...,"[-0.00676727294921875, -0.0186614990234375, -0..."
3,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,High-levelAPI|Keras|Pytorch**Low-levelAPIs**in...,"[-0.020355224609375, -0.0113983154296875, 0.00..."
4,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,"neuralnetworksasa**sequenceoflayers**,andmakec...","[-0.004070281982421875, 0.0033168792724609375,..."


## Retrieval: Vector Search & Similarity Index
|

In [9]:
from sklearn.neighbors import NearestNeighbors

all_embeddings=list(flattened_df['embeddings'])

all_embeddings_array = np.asarray(all_embeddings, dtype=np.float32)
nbrs = NearestNeighbors(n_neighbors=3, algorithm="ball_tree").fit(all_embeddings_array)

distances, indices=nbrs.kneighbors(all_embeddings_array)

flattened_df['indices']=indices.tolist()
flattened_df['distance'] = distances.tolist()

flattened_df.head()


,path,text,chunks,embeddings,indices,distance
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,#NeuralNetworkFrameworksAswehavelearnedalready...,"[0.01087188720703125, -0.0132904052734375, 0.0...","[0, 1, 26]","[0.0, 0.7877154999115473, 0.8570612873950955]"
1,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,hadtomanuallyprogramallderivativefunctionsinsi...,"[0.0180816650390625, -0.025421142578125, 0.057...","[1, 0, 26]","[0.0, 0.7877154999115473, 0.830694890200732]"
2,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,>✅Theterm'parallelize'meanstodistributethecomp...,"[-0.00676727294921875, -0.0186614990234375, -0...","[2, 7, 3]","[0.0, 0.8320083211873501, 0.8591954423438558]"
3,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,High-levelAPI|Keras|Pytorch**Low-levelAPIs**in...,"[-0.020355224609375, -0.0113983154296875, 0.00...","[3, 2, 7]","[0.0, 0.8591954423438558, 0.8642413057499462]"
4,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,"neuralnetworksasa**sequenceoflayers**,andmakec...","[-0.004070281982421875, 0.0033168792724609375,...","[4, 5, 0]","[0.0, 0.7424597971791285, 0.8915167069358068]"


In [10]:
question = "what is a perceptron ?"
query_embed=create_embeddings(question)

query_vector = np.asarray([query_embed], dtype=np.float32)

distances, indices = nbrs.kneighbors(query_vector, n_neighbors=3)

for i, idx in enumerate(indices[0]):
    print(f"--- Result {i+1} (Distance: {distances[0][i]:.4f})")
    print(f"Source:{flattened_df['path'].iloc[idx]}")
    print(flattened_df['chunks'].iloc[idx])
    print()


--- Result 1 (Distance: 0.9697)
Source:data/own_framework.md
#IntroductiontoNeuralNetworks.Multi-LayeredPerceptronIntheprevioussection,youlearnedaboutthesimplestneuralnetworkmodel-one-layeredperceptron,alineartwo-classclassificationmodel.Inthissectionwewillextendthismodelintoamoreflexibleframework,allowingusto:*perform**multi-classclassification**inadditiontotwo-class*solve**regressionproblems**inadditiontoclassification*separateclassesthatarenotlinearlyseparable

--- Result 2 (Distance: 1.0329)
Source:data/own_framework.md
supposedtobecalculatedconsideringthewholedataset(rememberthatlossiscalculatedasasumthroughalltrainingsamples).However,inreallifewetakesmallportionsofthedatasetcalled**minibatches**,andcalculategradientsbasedonasubsetofdata.Becausesubsetistakenrandomlyeachtime,suchmethodiscalled**stochasticgradientdescent**(SGD).##Multi-LayeredPerceptronsandBackpropagationOne-layernetwork,aswehaveseenabove,iscapable

--- Result 3 (Distance: 1.0460)
Source:data/own_framework.md
forbui

## Putting It All Together: RAG Chatbot with OpenAI

In [12]:
from typing import cast
from openai.types.chat import ChatCompletionMessageParam

def chatbot(user_input):
    query_vector_embed=create_embeddings(user_input)

    query_vector = np.asarray(query_vector_embed, dtype=np.float32).reshape(1, -1)
    distances, indices = nbrs.kneighbors(query_vector, n_neighbors=3)

    context_text=""
    for idx in indices[0]:
        context_text+=flattened_df['chunks'].iloc[idx]+ "\n\n"

    messages =[
            {"role": "system", "content": "You are an AI assistant that answers questions accurately using only the provided context."},
            {"role": "user", "content": f"Context:\n{context_text}\n\nQuestion: {user_input}"}
        ]

    messages = cast(
        list[ChatCompletionMessageParam],
        [
            {"role": "system", "content": "You are an AI assistant that answers questions accurately using only the provided context."},
            {"role": "user", "content": f"Context:\n{context_text}\n\nQuestion: {user_input}"},
        ],
    )

    response = client.chat.completions.create(
        model=chat_model,
        messages=messages,
        temperature=0.3,
        max_tokens=500
        
    )

    return response.choices[0].message.content

print(chatbot("what is perceptron?"))

A perceptron is the simplest type of neural network model, specifically a one-layered perceptron, which is used for linear two-class classification tasks. It serves as a foundational building block for more complex neural network architectures, such as multi-layered perceptrons, which can handle multi-class classification, regression problems, and separate classes that are not linearly separable.


In [14]:
from sklearn.metrics  import average_precision_score

test_cases = [
    {
        "query": "What is a perceptron?",
        "relevant_responses": ["A perceptron is a type of artificial neuron.", "It's a binary classifier used in machine learning."],
        "irrelevant_responses": ["A perceptron is a type of fruit.", "It's a type of car."]
    },
    {
        "query": "What is machine learning?",
        "relevant_responses": ["Machine learning is a method of data analysis that automates analytical model building.", "It's a branch of artificial intelligence."],
        "irrelevant_responses": ["Machine learning is a type of fruit.", "It's a type of car."]
    }
]

total_average_precision=0

for test_case in test_cases:
    query=test_case["query"]
    relevant_responses = test_case["relevant_responses"]
    irrelevant_responses = test_case["irrelevant_responses"]

    response=chatbot(query)

    all_responses=relevant_responses+irrelevant_responses
    true_labels= [1]*len(relevant_responses)+[0]*len(irrelevant_responses)

    predicted_scores=[1 if resp==response else 0 for resp in all_responses]

    average_precision=average_precision_score(true_labels,predicted_scores)

    total_average_precision +=average_precision

mean_average_precision = total_average_precision/len(test_cases)



In [15]:
print(f"mean_average_precision(map):{mean_average_precision}")

mean_average_precision(map):0.5
